# v060_d3_cap_4000 — LightGBM boosting cap 2000 → 4000 on v040's features

| Field | Value |
|---|---|
| **Version** | `v060_d3_cap_4000` |
| **Plan group** | D3 (LightGBM coordinate search, TRACKER #18) |
| **Parent version** | `v040` (`v040_idf_freq_features`) |
| **Author** | M4 Tarakesh |
| **Date** | 2026-09-26 |
| **Status** | running |
| **Hypothesis** | v040 stopped at 1999/2000 rounds; raising `n_estimators` to 4000 lets early stopping decide and raises tune F0.5 by ≥ 0.002. |

v040's pipeline with one LightGBM parameter changed. Everything upstream of the matcher (the
normalised records, the learned token map, the candidate pairs of every side and their 70
features) is read from a **feature snapshot** (`snapshot.py`, key `b25374ed`) built once with
v040's configuration by the pipeline's own functions, so every difference in the scores comes
from the model:

```
raw TSV -> normalise -> blocking -> v040's 70 features        (as v040, read from the snapshot)
        -> LightGBM, n_estimators 4000 (early stopping 100)   <- this version
        -> rule retuned on the tune split -> val and harder scored once with it frozen
```

`snapshot.evaluate_params` makes the same calls as `pipeline.fit` + `run_fold`
(`Matcher.fit` on the fit sample with early stopping on the stop sample, `decision.tune` on all
tune S1, `decide_by_country` and `score_pairs` on val); `tests/test_snapshot.py` checks that the
defaults reproduce the pipeline exactly, and `dataset/.cache/m4/repro_v040.py` checks it on the
real snapshot against v040's committed numbers (§5.3 shows both).

**Making v061+ from this notebook** (D3 is one axis per version): create the folder with
`python -m entity_resolution.tracking new <slug> --number 61`, copy this notebook over the
template copy (same file name as the folder), then edit only

1. this header table and §1 (hypothesis),
2. the **parameters cell of §2** (`VERSION`, `PARENT`, `PARAM_CHANGES`, `CHANGE`, `HYPOTHESIS`,
   and `SNAPSHOT_KEY` / `FEATURE_GROUPS` if the parent's data differ),
3. §8 (conclusion) after the run.

Nothing else refers to a version number. House rules: every code cell is preceded by a markdown
cell saying what it does and why; the decision metric is macro F0.5 on the fixed validation fold.

## 1. Hypothesis

* **Change vs parent:** `MatcherParams.n_estimators` 2000 → 4000. Every other parameter
  (`num_leaves` 63, `learning_rate` 0.05, `early_stopping` 100, `feature_fraction` 0.8, bagging
  0.8 / 1, `min_data_in_leaf` 200, `lambda_l2` 1.0, `max_bin` 255, seed 42, 12 threads), the
  features, the training sample and the rule grid are v040's.
* **Why it should raise macro F0.5:** v040's early stopping never fired: `best_iteration` 1999 of
  2000 means the stop-sample logloss was still falling when the cap ended training, so the model
  is under-fitted by construction. With the cap at 4000, early stopping (100 rounds without a
  stop-logloss gain) picks the number of rounds, which is what the D3 plan requires before any
  other axis is searched (a cap-limited model makes every later comparison a comparison of caps).
  Better-ranked probabilities should separate the same-name decoys and dropped-component true
  pairs that v040 still gets wrong (false merges 5,021, misses 46,202).
* **Prediction (falsifiable):** tune F0.5 ≥ v040 + 0.002 (v040: 0.98717) and val macro F0.5 >
  v040 + 0.002 (0.98698), harder-val not below v040 (0.98643), no slice down by more than 0.01;
  `best_iteration` < 4000 (early stopping decides). Stop-sample logloss lower than the same
  machine's re-run of v040.
* **Risk:** the extra rounds may buy little: M1's v101 (different features, cap 4000) stopped at
  1,666 rounds, and the rounds after 2000 may improve logloss far below what moves a threshold.
  Fit time grows roughly with the rounds used (up to 2×) and so does scoring time (more trees
  per prediction), which matters at test scale (§5.3 cost table).
* **Selection rule (D3, 08 §9):** the axis value is chosen by **tune** F0.5 after
  `decision.tune`, ties broken by stop logloss; val is scored once. KEEP / DROP / INVESTIGATE is
  13 §3 against the parent's committed `metrics.json` (§7).

## 2. Setup

All imports first. The cell after holds the parameters of this version; the one after that
derives the folders, reads the parent's `metrics.json` (the reference of every comparison, never
retyped) and finds the snapshot.

In [ ]:
import json
import time
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from entity_resolution import config as C
from entity_resolution.evaluate import entity_counts, entity_f05_from_counts, pair_in, slice_report
from entity_resolution.features import DEFAULT_GROUPS, FEATURE_COLUMNS
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import PipelineConfig, peak_rss_gb
from entity_resolution.snapshot import MANIFEST, SIDES, evaluate_params, load_snapshot
from entity_resolution.tracking import git_commit, log_result

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", 30)
np.random.seed(C.SEED)  # nothing below draws at random; LightGBM uses MatcherParams.seed
t_start = time.time()   # notebook wall time, printed at the end

### Parameters of this version (the only code cell a follow-up version edits)

* `PARAM_CHANGES` is the D3 axis this version moves; `PARAMS` is `MatcherParams()` (v040's
  model) with those fields replaced, so the diff against the parent is exactly `PARAM_CHANGES`
  (§4 checks it).
* `SNAPSHOT_KEY` names the snapshot of the parent's data; `FEATURE_GROUPS` are the parent's
  groups under today's registry names. v040 logged `frequency`, which was renamed `token_freq`
  on merge (`frequency` now names v101's core-name rates); `PARENT_GROUP_RENAMES` maps the
  parent's logged names so the check of the next cell compares like with like.
* `SEEDS = None` trains one model with `PARAMS.seed`; a tuple (42, 43, 44) averages one model
  per seed (the v068 step of the D3 plan).

In [ ]:
VERSION = "v060_d3_cap_4000"       # this folder under experiments/
PARENT = "v040_idf_freq_features"  # logged version compared against (its committed metrics.json)
GROUP = "D3"                       # plan ID: LightGBM coordinate search (TRACKER #18)
OWNER = "M4"
SNAPSHOT_KEY = "b25374ed"          # dataset/.cache/m4/<key>/: the parent's data, featured once
FEATURE_GROUPS = DEFAULT_GROUPS + ("idf", "token_freq", "ctx_idf", "address_extra")  # v040's 12
PARENT_GROUP_RENAMES = {"frequency": "token_freq"}  # parent's logged group name -> registry name
PARAM_CHANGES = {"n_estimators": 4000}              # the one axis this version moves
PARAMS = replace(MatcherParams(), **PARAM_CHANGES)  # every other field: v040's defaults
SEEDS = None                       # None: one model; (42, 43, 44): seed average
CHANGE = "n_estimators 2000 -> 4000 on v040 features (snapshot b25374ed)"
HYPOTHESIS = ("v040 stopped at 1999/2000 rounds; raising n_estimators to 4000 lets early "
              "stopping decide and raises tune F0.5 by >= 0.002")

# the pipeline configuration of this version: its grid is the one decision.tune searches
cfg = PipelineConfig(feature_groups=FEATURE_GROUPS, model=PARAMS)

Derived paths, the parent's record and the snapshot folder. `find_snapshot` takes
`SNAPSHOT_KEY` when its manifest lists exactly `FEATURE_GROUPS` and all five sides are built;
otherwise (a code change re-keyed the snapshot) the newest complete snapshot with those groups,
with a warning. The parent's logged groups, renamed, must equal `FEATURE_GROUPS`, or the
comparisons below would not isolate the model change. `REPRO_DIR` holds the same-machine re-run
of the parent on this snapshot (`repro_v040.py`), used when the parent's own artifacts are not
on this machine (they are gitignored).

In [ ]:
EXP_DIR = C.EXPERIMENTS / VERSION
ARTIFACTS = EXP_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
PARENT_DIR = C.EXPERIMENTS / PARENT
PARENT_ARTIFACTS = PARENT_DIR / "artifacts"  # gitignored: only on the machine that ran it
PARENT_TAG = PARENT.split("_")[0]            # "v040": the csv's parent column
parent_record = json.loads((PARENT_DIR / "metrics.json").read_text(encoding="utf-8"))
parent_metrics = parent_record["metrics"]    # the parent's 13 §2.2 record
M4_CACHE = C.DATASET / ".cache" / "m4"


def find_snapshot(key: str, groups: tuple[str, ...]) -> Path:
    '''Folder of the snapshot to evaluate.

    ``key`` when its manifest lists exactly ``groups`` and every side is built; else the
    newest such folder under ``M4_CACHE`` (a changed data module re-keys a snapshot), with a
    warning; FileNotFoundError when there is none.
    '''
    def usable(folder: Path) -> bool:
        '''Manifest present, same feature groups, all sides built.'''
        path = folder / MANIFEST
        if not path.exists():
            return False
        m = json.loads(path.read_text(encoding="utf-8"))
        return (m["key_parts"]["feature_groups"] == list(groups)
                and all(s in m["sides"] for s in SIDES))

    if usable(M4_CACHE / key):
        return M4_CACHE / key
    found = [p for p in M4_CACHE.iterdir() if p.is_dir() and usable(p)]
    if not found:
        raise FileNotFoundError(f"no complete snapshot with groups {groups} under {M4_CACHE}")
    newest = max(found, key=lambda p: (p / MANIFEST).stat().st_mtime)
    print(f"WARNING: snapshot {key} missing or incomplete; using {newest.name}")
    return newest


SNAP_DIR = find_snapshot(SNAPSHOT_KEY, FEATURE_GROUPS)
REPRO_DIR = SNAP_DIR / f"repro_{PARENT_TAG}"  # repro_v040.py's slices and val matches
parent_groups = [PARENT_GROUP_RENAMES.get(g, g) for g in parent_metrics["feature_groups"]]
if parent_groups != list(FEATURE_GROUPS):
    raise ValueError(f"parent groups {parent_groups} differ from FEATURE_GROUPS {FEATURE_GROUPS}")
print(f"{VERSION}: parent {parent_record['version']} (local F0.5 {parent_metrics['f_beta']:.4f}, "
      f"decision {parent_record['decision']}); snapshot {SNAP_DIR}; code at {git_commit()}")

## 3. Data

The fixed validation split (`split.load_fold`, 20 % of Source 1 by id hash, seed 42) and the
train fold's inner split, exactly as v040 used them, but read from the snapshot rather than
rebuilt: **fit** (200k fit-side S1 with all their candidates, the training rows), **stop** (50k
tune-side S1, early stopping), **tune** (every tune-side S1, the rule grid), **val** (the fixed
val fold, scored once) and **harder** (`evaluate.harder_fold(val)`: 20 % of the val S1 dropped,
their pool records kept as decoys). `load_snapshot` reads the manifest only; each side is read
from Parquet when it is used, one at a time. The table shows each side's size and candidate
recall as recorded when it was built: val must show v040's `cand_recall` 0.9906.

In [ ]:
snap = load_snapshot(SNAP_DIR)  # manifest only: sides are streamed on demand
if missing := [s for s in SIDES if s not in snap.sides]:
    raise ValueError(f"snapshot {snap.key} lacks sides {missing}")
manifest = snap.manifest
sides_table = pd.DataFrame({
    side: {"rows": st["rows"], "positives": st["positives"], "s1": st["s1"],
           "MB": round(st["bytes"] / 1e6),
           "pair_recall": (st.get("blocking") or {}).get("pair_recall"),
           "cands_mean": (st.get("blocking") or {}).get("candidates_mean")}
    for side, st in manifest["sides"].items()}).T
print(f"snapshot {snap.key}: {len(snap.columns)} features, token map "
      f"{manifest['token_map_size']} tokens, built {manifest['created']} at commit "
      f"{manifest['commit']}")
sides_table

## 4. Method

### 4.1–4.3 Normalisation, blocking, pair features (as in parent)

v040's `NormaliseConfig`, learned token map, `BlockingConfig` (exact keys, name char 3-grams,
name + address word TF-IDF, 60 candidates per S1 per country) and its 12 feature groups
(v001's 47 features + `idf`, `token_freq`, `ctx_idf`, `address_extra`: 70 columns), all frozen
in the snapshot. The cell below compares the snapshot's blocking configuration and this
version's model parameters with the parent's logged ones: blocking must be identical (fields
added to the library since the parent ran are listed apart), and the model must differ in
exactly `PARAM_CHANGES`.

In [ ]:
def config_diff(ours: dict, theirs: dict) -> tuple[dict, dict]:
    '''``(changed, added)`` between two JSON-ready configurations.

    ``changed``: keys both hold with different values, {key: (ours, theirs)}; ``added``: keys
    only ``ours`` holds (fields added to the library after ``theirs`` was logged), {key: ours}.
    '''
    changed = {k: (ours[k], theirs[k]) for k in ours if k in theirs and ours[k] != theirs[k]}
    added = {k: ours[k] for k in ours if k not in theirs}
    return changed, added


ours_model = json.loads(json.dumps(asdict(PARAMS)))  # JSON round trip, as the parent's record
defaults = asdict(MatcherParams())
for stage, ours, theirs in (("blocking", manifest["config"]["blocking"],
                             parent_metrics["blocking_config"]),
                            ("model", ours_model, parent_metrics["model_params"])):
    changed, added = config_diff(ours, theirs)
    print(f"{stage}: changed {changed or 'nothing'}; fields the parent's record lacks: {added}")
    if stage == "blocking" and changed:
        raise ValueError("the snapshot's blocking differs from the parent's: wrong snapshot")
    if stage == "model":
        if set(changed) != set(PARAM_CHANGES):
            raise ValueError(f"model differs from the parent in {sorted(changed)}, "
                             f"expected exactly {sorted(PARAM_CHANGES)}")
        if any(v != defaults[k] for k, v in added.items()):
            raise ValueError(f"a field the parent lacks is not at its default: {added}")

### 4.4 Matching model

LightGBM (MIT licence, a few MB of trees: far below the 8B-parameter limit), trained on the fit
sample's 6.72M candidate pairs (positive rate about 0.10) with early stopping on the stop
sample's logloss, no class reweighting (`scale_pos_weight` 1.0), CPU, 12 threads, seed 42.
Only `PARAM_CHANGES` differs from v040 (table below).

### 4.5 Decision rule (as in parent, retuned on the tune split)

Same rule family and grid (`cfg.grid`, the pipeline default: pool-side 1-to-1, `tau_abs`,
`tau_rel`, `tau_single`, `max_matches`), grid-searched again for macro F0.5 on all tune-side S1:
new probabilities need new thresholds.

In [ ]:
params_table = pd.DataFrame({"parent": pd.Series(parent_metrics["model_params"], dtype=object),
                             "this": pd.Series(ours_model, dtype=object)})
params_table["changed"] = params_table.index.isin(list(PARAM_CHANGES))
display(params_table)
print("grid:", asdict(cfg.grid))

## 5. Evaluation on the validation fold

### 5.1 Train, tune the rule, score val and harder

`evaluate_params` trains the matcher (the fit matrix is freed once LightGBM has binned it),
scores the tune side and grid-searches the rule on it, freezes the rule, then scores val and
harder in 250k-row batches. It returns the 13 §2.2 metrics of this stage plus the M4 keys
(`best_iteration`, stop-sample `tune_logloss` / `tune_auc`, calibration `ece_tune`,
`brier_tune`, `ece_tune_rank1`, `ece_val`, `brier_val`, `importance_top20`,
`f_beta_by_country`), and fills `arts` with the tune table, reliability table, importance,
val matches and slice report. Expect about 10–25 min on the laptop (fit dominates; about 5–7 GB
peak). The scored val pairs are dropped at once: nothing below needs them.

In [ ]:
arts: dict = {}
t0 = time.time()
metrics, model, rule = evaluate_params(snap, PARAMS, grid=cfg.grid, seeds=SEEDS, artifacts=arts)
eval_seconds = time.time() - t0
arts.pop("val_scored", None)  # ~15M scored val pairs: not used below, freed now
print(f"evaluate_params {eval_seconds:.0f} s; peak RSS {peak_rss_gb()} GB")
print("training:", model.fit_info_)
print("rule:", asdict(rule), "| parent:", parent_metrics["rule"])

### 5.2 Artifacts

Written to `artifacts/` (gitignored), enough to regenerate every table of this notebook and to
serve as a child version's parent reference: the model (`model/`, with the full importance
series as `model/importance.csv`), `rule.json`, `config.json`, `tune_table.csv`,
`reliability.csv` (20 equal-count bins, tune and val, all pairs and rank-1 pairs),
`importance.csv` (gain share, rank, group) and `val_matches.parquet`. Then the 20 most important
features and the best rules of the tune grid.

In [ ]:
model.save(ARTIFACTS / "model")
(ARTIFACTS / "rule.json").write_text(json.dumps(asdict(rule), indent=2) + "\n", encoding="utf-8")
(ARTIFACTS / "config.json").write_text(
    json.dumps({**cfg.record(), "snapshot": snap.key, "seeds": SEEDS}, indent=2) + "\n",
    encoding="utf-8")
arts["tune_table"].to_csv(ARTIFACTS / "tune_table.csv", index=False)
arts["reliability"].to_csv(ARTIFACTS / "reliability.csv", index=False)
arts["val_matches"].to_parquet(ARTIFACTS / "val_matches.parquet", index=False)
group_of = {c: g for g in FEATURE_GROUPS for c in FEATURE_COLUMNS[g]}
importance = arts["importance"].rename("gain").to_frame()
importance["rank"] = np.arange(1, len(importance) + 1)  # 1 = largest gain
importance["group"] = importance.index.map(group_of)
importance.to_csv(ARTIFACTS / "importance.csv")
importance.to_csv(ARTIFACTS / "model" / "importance.csv")  # CLAUDE.md §8 location
display(importance.head(20))
print(f"zero-gain features: {list(importance.index[importance['gain'] == 0])}")
arts["tune_table"].sort_values("f_beta", ascending=False).head(10)

### 5.3 Comparison with the parent

`parent` is v040's committed `metrics.json` (M3's pipeline run): the reference of the 13 §3
decision. `parent_repro` is the parent's configuration re-run on this snapshot on this machine
(`repro_v040.py`, `REPRO_DIR/../repro_metrics.json`), shown when it exists: it separates the
effect of this version's change (`delta_repro`) from machine and snapshot drift, and it supplies
the parent's stop logloss / AUC and calibration, which v040 did not log. `cand_recall` is a
sanity row (same candidates, must not move). The second table puts run time and peak RSS next to
the same-machine re-run when there is one (13 §3: a doubled run time or RSS is a reason to
INVESTIGATE).

In [ ]:
COMPARE_KEYS = ("f_beta", "tune_f_beta", "harder_f_beta", "f_beta_singletons", "f_beta_matched",
                "pair_precision", "pair_recall", "cand_recall", "n_fp", "n_fn",
                "n_false_singleton", "best_iteration", "tune_logloss", "tune_auc", "ece_tune",
                "brier_tune")
COST_KEYS = ("load_seconds", "fit_seconds", "tune_seconds", "score_seconds", "decide_seconds",
             "peak_rss_gb")
repro_path = SNAP_DIR / "repro_metrics.json"
repro = {}
if repro_path.exists():
    repro_record = json.loads(repro_path.read_text(encoding="utf-8"))
    if Path(repro_record["parent"]).parent.name == PARENT:  # a re-run of this parent
        repro = repro_record["metrics"]
print(f"same-machine parent re-run: {'read from ' + str(repro_path) if repro else 'unavailable'}")


def as_float(value) -> float:
    '''A logged number as float; NaN for a missing key or a non-number.'''
    return float(value) if isinstance(value, (int, float)) else float("nan")


def versus_parent(keys: tuple[str, ...]) -> pd.DataFrame:
    '''Parent (logged), parent re-run on this snapshot, and this version, per key, with deltas.'''
    table = pd.DataFrame({
        "parent": [as_float(parent_metrics.get(k)) for k in keys],
        "parent_repro": [as_float(repro.get(k)) for k in keys],
        "this": [as_float(metrics.get(k)) for k in keys]}, index=list(keys))
    return table.assign(delta=table["this"] - table["parent"],
                        delta_repro=table["this"] - table["parent_repro"])


comparison = versus_parent(COMPARE_KEYS)
display(comparison.round(5))
costs = versus_parent(COST_KEYS)
reference = costs["parent_repro"].fillna(costs["parent"])  # same machine when available
costs = costs.assign(ratio=costs["this"] / reference)[["parent", "parent_repro", "this", "ratio"]]
costs.round(2)

### 5.4 Slices against the parent

This version's slice report (`evaluate.slice_report`, 11 §7: country, source, non-Latin,
ambiguous names, singletons, number of true matches, postcode, empty addresses) next to the
parent's, on the same val fold and normalised records. The parent's slices are recomputed from
its `val_matches.parquet` when the parent's artifacts or the same-machine re-run have one, else
read from a `slices.csv` there, else unavailable (the output says which). `max_slice_drop` is
the largest `parent − this` slice F0.5 (negative when every slice rose), NaN when unavailable,
which §7 treats as "no drop". Written to `artifacts/slices.csv` and
`artifacts/slices_vs_parent.csv`.

In [ ]:
val = snap.fold("val")                  # ids-only val fold: S1 with country, pool ids, truth
s1n_val, pooln_val = snap.normalised()  # normalised val S1 and matched pool rows
slices = arts["slices"]
slices.to_csv(ARTIFACTS / "slices.csv", index=False)


def parent_reference() -> tuple[pd.DataFrame | None, pd.DataFrame | None, str]:
    '''(parent val matches or None, parent slices or None, where they came from).

    Looks in the parent's artifacts first, then in the same-machine re-run's folder; val
    matches beat a slices.csv because they also give the pair transitions of §6.2.
    '''
    for folder in (PARENT_ARTIFACTS, REPRO_DIR):
        matches_path, slices_path = folder / "val_matches.parquet", folder / "slices.csv"
        if matches_path.exists():
            pm = pd.read_parquet(matches_path)
            return pm, slice_report(pm, val, s1n_val, pooln_val), f"recomputed from {matches_path}"
        if slices_path.exists():
            ps = pd.read_csv(slices_path, dtype={"family": str, "slice": str},
                             keep_default_na=False, na_values=[""])  # "0" stays a slice name
            return None, ps, f"read from {slices_path}"
    return None, None, f"UNAVAILABLE (nothing in {PARENT_ARTIFACTS} or {REPRO_DIR})"


parent_matches, parent_slices, source = parent_reference()
slice_cmp = slices[["family", "slice", "entities", "f_beta", "pair_precision", "pair_recall"]]
if parent_slices is None:
    slice_cmp = slice_cmp.assign(f_beta_parent=np.nan)
else:
    slice_cmp = slice_cmp.merge(
        parent_slices[["family", "slice", "f_beta"]].rename(columns={"f_beta": "f_beta_parent"}),
        on=["family", "slice"], how="left")
slice_cmp["delta"] = slice_cmp["f_beta"] - slice_cmp["f_beta_parent"]
drop = -slice_cmp["delta"]            # > 0: the slice fell
max_slice_drop = float(drop.max())    # NaN when no parent slice is known
slice_cmp.to_csv(ARTIFACTS / "slices_vs_parent.csv", index=False)
print(f"parent slices: {source}")
if np.isnan(max_slice_drop):
    print("max_slice_drop = NaN: the slice condition of the 13 §3 rule is skipped in §7")
else:
    worst = slice_cmp.loc[drop.idxmax()]
    print(f"max_slice_drop = {max_slice_drop:+.4f} (worst: {worst['family']} / {worst['slice']}, "
          f"{worst['entities']} entities)")
print("F0.5 by country:", {k: round(v, 5) for k, v in metrics["f_beta_by_country"].items()})
slice_cmp[["family", "slice", "entities", "f_beta_parent", "f_beta", "delta", "pair_precision",
           "pair_recall"]]

### 5.5 Calibration (08 §5)

Expected calibration error and Brier score of the tune-side probabilities (20 equal-count bins),
the same on the rank-1 pairs (`ctx_rank_name == 1`, the pairs the decision layer looks at
first) and on val. An isotonic calibrator is added only if `ece_tune > 0.02` **and** the
decision layer needs probabilities (it does not today: the rule is retuned per version). The
tune table over all pairs follows (`artifacts/reliability.csv` has all four).

In [ ]:
CALIB_KEYS = ("ece_tune", "brier_tune", "ece_tune_rank1", "ece_val", "brier_val")
print({k: round(metrics[k], 5) for k in CALIB_KEYS if k in metrics},
      "| isotonic needed:", metrics.get("ece_tune", 0.0) > 0.02)
rel = arts["reliability"]
rel[(rel["side"] == "tune") & (rel["subset"] == "all")].round(4)

## 6. Error analysis

### 6.1 Error kinds

Pair counts of the four kinds (`snapshot.error_counts`, equal to v001's `error_samples` counts):
false merges on matched entities, missed true pairs of entities that predicted something,
matched entities predicted empty (false singletons) and predictions on true singletons; each
next to the parent's logged count and the same-machine re-run's. The snapshot holds ids, not
raw records, so samples with names and addresses come from the parent's §6 or from
`evaluate.error_samples` on a raw val fold if a pattern needs a look.

In [ ]:
KINDS = ("false_merge", "missed", "false_singleton", "singleton_merge")
repro_errors = (json.loads(repro_path.read_text(encoding="utf-8")).get("errors") or {}
                if repro else {})
errors_table = pd.DataFrame({
    "parent": [parent_metrics.get("errors", {}).get(k, np.nan) for k in KINDS],
    "parent_repro": [repro_errors.get(k, np.nan) for k in KINDS],
    "this": [metrics["errors"][k] for k in KINDS]}, index=list(KINDS))
errors_table.assign(delta=errors_table["this"] - errors_table["parent"],
                    delta_repro=errors_table["this"] - errors_table["parent_repro"])

### 6.2 Transitions from the parent

When the parent's val matches are available (§5.4): the pairs this version fixed (a parent false
merge dropped, a parent miss now predicted) and broke (a new false merge, a parent true pair
lost), and the entities whose F0.5 rose or fell. Skipped, with a message, otherwise.

In [ ]:
val_matches = arts["val_matches"]
if parent_matches is None:
    print("transitions skipped: no parent val_matches.parquet on this machine")
else:
    this_true = pair_in(val_matches, val.pairs)
    this_in_parent = pair_in(val_matches, parent_matches)
    parent_true = pair_in(parent_matches, val.pairs)
    parent_in_this = pair_in(parent_matches, val_matches)
    display(pd.Series({
        "fixed: parent false merge dropped": int((~parent_true & ~parent_in_this).sum()),
        "fixed: parent miss now predicted": int((this_true & ~this_in_parent).sum()),
        "broken: new false merge": int((~this_true & ~this_in_parent).sum()),
        "broken: parent true pair lost": int((parent_true & ~parent_in_this).sum()),
    }, name="pairs").to_frame())
    delta = (entity_f05_from_counts(*entity_counts(val_matches, val))
             - entity_f05_from_counts(*entity_counts(parent_matches, val)))
    print(f"entities: {int((delta > 0).sum()):,} improved, {int((delta < 0).sum()):,} worsened; "
          f"macro F0.5 {delta.mean():+.5f}")

## 7. Log the result

Records `metrics.json` and this version's row in `experiments/experiments.csv`, stamped with the
git commit of `src/` (it must not end in `-dirty`). The decision is the 13 §3 rule, computed
against the parent's committed record:

* **KEEP**: `f_beta` > parent + 0.002, `max_slice_drop` ≤ 0.01 (or unavailable) and
  `harder_f_beta` ≥ the parent's;
* **DROP**: `f_beta` ≤ parent, or a slice down by more than 0.01;
* **INVESTIGATE**: anything else, and a would-be KEEP with a mixed signal: singleton and matched
  F0.5 moving in opposite directions by more than 0.001, or val `score_seconds` / peak RSS more
  than doubled against the same-machine reference (fit time is reported in §5.3, judged by hand
  in §8: more rounds are this version's expected price).

The record holds the 13 §2.2 keys (stages the snapshot does not re-run, normalisation, blocking
and features, have no timing key), the M4 keys (`best_iteration`, `tune_logloss`, `tune_auc`,
`ece_tune`, `brier_tune`, `importance_top20`) and distinct extras (`ece_tune_rank1`, `ece_val`,
`brier_val`, `f_beta_by_country`, `max_slice_drop`, `snapshot`, `seeds`, `token_map_size`).

In [ ]:
MARGIN, SLICE_TOL, MIXED_TOL = 0.002, 0.01, 0.001  # 13 §3 margin and slice drop; mixed signal
f_beta, parent_f = metrics["f_beta"], parent_metrics["f_beta"]
harder = metrics.get("harder_f_beta", float("nan"))
parent_harder = as_float(parent_metrics.get("harder_f_beta"))  # NaN: no harder check
slices_ok = np.isnan(max_slice_drop) or max_slice_drop <= SLICE_TOL
harder_ok = np.isnan(parent_harder) or harder >= parent_harder
d_single = comparison.loc["f_beta_singletons", "delta"]
d_matched = comparison.loc["f_beta_matched", "delta"]
warnings = []
if min(d_single, d_matched) < -MIXED_TOL and max(d_single, d_matched) > 0:
    warnings.append(f"singletons {d_single:+.4f} vs matched {d_matched:+.4f}")
for k in ("score_seconds", "peak_rss_gb"):
    if costs.loc[k, "ratio"] > 2:
        warnings.append(f"{k} x{costs.loc[k, 'ratio']:.1f}")
if f_beta > parent_f + MARGIN and slices_ok and harder_ok:
    DECISION = "INVESTIGATE" if warnings else "KEEP"
elif f_beta <= parent_f or not slices_ok:
    DECISION = "DROP"
else:
    DECISION = "INVESTIGATE"
print(f"f_beta {f_beta:.5f} vs parent {parent_f:.5f} ({f_beta - parent_f:+.5f}; KEEP needs "
      f"> +{MARGIN}); max_slice_drop {max_slice_drop:.4f} (<= {SLICE_TOL}); harder "
      f"{harder:.5f} vs {parent_harder:.5f}; warnings {warnings or 'none'} -> {DECISION}")

SCORE_KEYS = ("f_beta", "f_beta_singletons", "f_beta_matched", "pair_precision", "pair_recall")
BLOCK_KEYS = ("cand_recall", "entity_recall", "ceiling_f_beta", "cands_mean", "cands_p95")
TIME_KEYS = ("load_seconds", "fit_seconds", "tune_seconds", "score_seconds", "decide_seconds")
M4_KEYS = ("best_iteration", "tune_logloss", "tune_auc", "ece_tune", "brier_tune",
           "importance_top20")
EXTRA_KEYS = ("ece_tune_rank1", "ece_val", "brier_val", "f_beta_by_country")
record = {
    "hypothesis": HYPOTHESIS,
    "blocking_config": manifest["config"]["blocking"], "feature_groups": list(FEATURE_GROUPS),
    "model_params": asdict(PARAMS), "rule": asdict(rule),
    **{k: metrics[k] for k in SCORE_KEYS + BLOCK_KEYS},
    "harder_f_beta": harder, "tune_f_beta": metrics["tune_f_beta"],
    "n_fp": metrics["n_fp"], "n_fn": metrics["n_fn"],
    "n_false_singleton": metrics["n_false_singleton"], "errors": metrics["errors"],
    **{k: metrics[k] for k in TIME_KEYS},
    "peak_rss_gb": max(metrics["peak_rss_gb"], peak_rss_gb()), "decision": DECISION,
    **{k: metrics[k] for k in M4_KEYS + EXTRA_KEYS if k in metrics},
    "max_slice_drop": None if np.isnan(max_slice_drop) else max_slice_drop,
    "snapshot": snap.key, "seeds": metrics["seeds"],
    "token_map_size": manifest["token_map_size"],
}
slice_note = "n/a" if np.isnan(max_slice_drop) else f"{max_slice_drop:+.4f}"
row = log_result(
    EXP_DIR, change=CHANGE, group=GROUP, local_f05=f_beta, cand_recall=metrics["cand_recall"],
    notes=(f"snapshot {snap.key}; best_iteration {metrics['best_iteration']}/"
           f"{PARAMS.n_estimators}; tune F0.5 {metrics['tune_f_beta']:.4f} "
           f"({metrics['tune_f_beta'] - parent_metrics['tune_f_beta']:+.4f}); singleton F0.5 "
           f"{metrics['f_beta_singletons']:.4f}; harder {harder:.4f}; "
           f"{f_beta - parent_f:+.4f} vs {PARENT_TAG}; max slice drop {slice_note}"),
    owner=OWNER, parent=PARENT_TAG, decision=DECISION, metrics=record)
if row["commit"].endswith("-dirty"):
    print("WARNING: src/ has uncommitted changes: commit them and re-run before committing this")
row

## 8. Conclusion

*(Filled in after the run from the summary printed below and the tables of §5–§6.)*

* **Result vs parent (§5.3):** local macro F0.5 … → … (Δ …); tune …; harder …; singletons /
  matched …; same-machine re-run of v040 …
* **Early stopping:** `best_iteration` … of 4000; stop logloss … (v040 re-run …).
* **Decision (§7):** …
* **Slices (§5.4) and errors (§6):** …
* **Cost (§5.3):** fit … s, val scoring … s, peak RSS … GB (ratios vs the re-run).
* **Next (D3 queue):** if KEEP, v061 `num_leaves` {31, 127} on this parent with the cap at
  4000; if DROP, the cap stays 4000 anyway only if M1 agrees (early stopping must decide), and
  v061 moves `num_leaves` on v040.

In [ ]:
cmp = comparison
cap_hit = metrics["best_iteration"] >= PARAMS.n_estimators - 1
stopping = "cap hit" if cap_hit else "early stopping decided"
repro_logloss = cmp.loc["tune_logloss", "parent_repro"]
print(
    f"{VERSION} ({CHANGE}): local macro F0.5 {cmp.loc['f_beta', 'parent']:.5f} -> "
    f"{f_beta:.5f} ({cmp.loc['f_beta', 'delta']:+.5f}; vs same-machine re-run "
    f"{cmp.loc['f_beta', 'delta_repro']:+.5f}); tune {metrics['tune_f_beta']:.5f} "
    f"({cmp.loc['tune_f_beta', 'delta']:+.5f}); harder {harder:.5f} "
    f"({cmp.loc['harder_f_beta', 'delta']:+.5f}); singletons {metrics['f_beta_singletons']:.5f} "
    f"({d_single:+.5f}), matched {metrics['f_beta_matched']:.5f} ({d_matched:+.5f}); pair P "
    f"{metrics['pair_precision']:.5f}, R {metrics['pair_recall']:.5f}. best_iteration "
    f"{metrics['best_iteration']} of {PARAMS.n_estimators} ({stopping}); "
    f"stop logloss {metrics['tune_logloss']:.5f} (re-run {repro_logloss:.5f}); "
    f"ece_tune {metrics.get('ece_tune', float('nan')):.4f}. Errors: "
    + ", ".join(f"{k} {int(errors_table.loc[k, 'parent'])} -> {int(errors_table.loc[k, 'this'])}"
                for k in KINDS)
    + f". Max slice drop {slice_note}. Decision {DECISION}. Cost: fit "
    f"{metrics['fit_seconds']:.0f} s "
    f"(x{costs.loc['fit_seconds', 'ratio']:.2f}), val+harder+tune scoring "
    f"{metrics['score_seconds']:.0f} s (x{costs.loc['score_seconds', 'ratio']:.2f}), peak RSS "
    f"{record['peak_rss_gb']:.2f} GB. Notebook {time.time() - t_start:.0f} s.")

## 9. Test inference (not for M4 versions)

M4 versions are never uploaded: only M1 uploads, and only integrated v1xx runs (13 §5). If this
change is KEEP, the PR carries `PARAM_CHANGES` into `MatcherParams` (or M1's v1xx config) and M1
re-runs the full pipeline. For a quick test-shaped check, `snap.to_fitted(cfg, model, rule,
arts["tune_table"])` gives a `pipeline.Fitted` that `pipeline.run_test` accepts.